In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [2]:
!pip install pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 54.5 MB/s  0:00:00


In [3]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kagglehub]/4 [kagglesdk]


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ultralytics/coco128")

print("Path to dataset files:", path)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 6.66M/6.66M [00:00<00:00, 7.21MB/s]

Extracting files...
Path to dataset files: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3


In [9]:
import os

dataset_root = "/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3"

for root, dirs, files in os.walk(dataset_root):
    print("DIR:", root)
    print("Subdirs:", dirs)
    print("Files:", files[:5])
    print("-" * 50)

DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3
Subdirs: ['coco128']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128
Subdirs: ['labels', 'images']
Files: ['README.txt', 'LICENSE']
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017
Subdirs: []
Files: ['000000000294.txt', '000000000036.txt', '000000000471.txt', '000000000491.txt', '000000000315.txt']
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/ka

In [10]:
import glob

img_dir = dataset_root + "/coco128/images/train2017"
lbl_dir = dataset_root + "/coco128/labels/train2017"

images = sorted(glob.glob(img_dir + "/*.jpg"))
labels = sorted(glob.glob(lbl_dir + "/*.txt"))

print("Images:", len(images))
print("Labels:", len(labels))
print(images[0])
print(labels[0])

Images: 128
Labels: 128
/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images/train2017/000000000009.jpg
/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017/000000000009.txt


In [11]:
len(images)

128

In [12]:
from dataloader import DataSSD300,DataSplitter

In [34]:
for image, labels_list, gt_box_list in train_dataloader:
    print(image.shape)
 
    print((gt_box_list[0].shape))
    print((labels_list[0].shape))
    print((labels_list[0].dtype))
    break

torch.Size([20, 3, 300, 300])
torch.Size([11, 4])
torch.Size([11])
torch.int64


In [14]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




1.3%

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/onyxia/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100.0%


ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [20]:
from utils import setup_device_and_seed

device = setup_device_and_seed(seed=42)

In [ ]:
from ssd import SSD
from multigpusetup import ddp_setup
from torch.distributed import init_process_group, destroy_process_group
from train import train 
import  torch.multiprocessing as mp 
def main(rank : int  ,nb_classes : int ,nb_gpus : int ):

    
    if nb_gpus==0:
         device = torch.device("cpu")
    elif nb_gpus==1:
        device = torch.device("cuda:0")
    elif nb_gpus>1:
        ddp_setup(rank, nb_gpus)
        device = torch.device(f"cuda:{rank}")
    else:
        raise ValueError("no nb gpus specified")

    coco128loader=DataSSD300(img_dir,lbl_dir,gt_normalised=True)
    splitter=DataSplitter(batch_size=20,test_size=0.15,val_size=0.15,multigpu = True if nb_gpus>1 else False)
    train_dataloader, val_dataloader, test_dataloader=splitter(coco128loader)


    model=SSD(base,nb_classes=nb_classes,phase="train",alpha=1,device=device,prob_thr=0.01,nms_thr=0.45,top_k=200,variances=(0.1,0.2)).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, weight_decay = 0.0005, momentum = 0.9)
    train(model,optimizer,train_dataloader,val_dataloader,modelname="ssd_coco128V1")
    if nb_gpus>1:
        destroy_process_group()

In [ ]:
nb_gpus=torch.cuda.device_count()
nb_classes=21
if nb_gpus>1:
    mp.spawn(main,args=(nb_classes,nb_gpus),nprocs=nb_gpus)
elif nb_gpus==1:
    main(None,nb_classes,1)
else:
    main(None,nb_classes,0)
